# Jumper su Kaggle — smoke test (setup + misura di velocità)

**Cosa fa.** Installa il repo `KingKongRobotics/jumper` (commit bloccato) in un ambiente isolato, verifica GPU/CUDA/Warp/EGL con il check ufficiale, poi lancia tre mini-training da 12 iterazioni e misura **env-step/s, secondi per iterazione, memoria GPU**. Niente viene allenato davvero: serve solo a dire *se gira* e *quanto va veloce* su questa macchina.

**Prima di premere Run (barra a destra, ⚙ Settings):**
1. **Accelerator → GPU T4 x2** (mai P100: non è supportata).
2. **Internet → On** (serve per clone e pip).
3. Lascia *Persistence* su Nothing: il repo e il venv vanno in `/kaggle/temp` (effimero), i risultati in `/kaggle/working/jumper-smoke/`.

**Poi:** *Run All*. Tempo totale ≈ 10–15 min (6–10 di installazione, ~5 di misure). Alla fine scarica `smoke.json` da `/kaggle/working/jumper-smoke/` (pannello Output a destra) e mandamelo, oppure incolla la tabella finale.

Se la cella 1 si ferma su *"GPU … non supportata"* hai la P100: cambia acceleratore e riavvia la sessione.


In [ ]:
# @title 1. Setup Kaggle: clone, venv, torch, EGL  (~6-10 min)
import json
import os
import signal
import subprocess
import sys
import venv
from collections import deque
from pathlib import Path

REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
# Commit bloccato: la ripresa dei checkpoint confronta revisione e sorgenti.
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"

ON_KAGGLE = Path("/kaggle").is_dir()
# Effimero: venv e repo NON finiscono nell'output del notebook (7 GB).
WORK = Path("/kaggle/temp") if ON_KAGGLE else Path("/tmp/jumper-work")
# Persistente: solo i risultati piccoli (json, tabelle) finiscono qui.
OUT = Path("/kaggle/working/jumper-smoke") if ON_KAGGLE else Path.cwd() / "jumper-smoke"
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
REPO = WORK / "jumper"

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Jumper vuole Python 3.10-3.13, qui c'è {sys.version}")


def run(command, *, cwd=None, check=True):
    """Esegue e mostra l'output in diretta; Ctrl-C arriva a tutto il gruppo."""
    print("+", " ".join(str(p) for p in command), flush=True)
    command = [str(p) for p in command]
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        code = process.wait()
        if code and check:
            raise subprocess.CalledProcessError(code, command, output="".join(tail))
        return subprocess.CompletedProcess(command, code, stdout="".join(tail))
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=15)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        raise
    finally:
        process.stdout.close()


# ── GPU: deve essere una T4 (o meglio). P100 = compute capability 6.0 → rifiutata ──
run(["nvidia-smi", "-L"])
rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"],
    text=True,
).strip().splitlines()
if not rows:
    raise RuntimeError("Nessuna GPU: in Kaggle apri Settings → Accelerator → GPU T4 x2")
gpus = [[p.strip() for p in r.split(",")] for r in rows]
for g in gpus:
    print("GPU", *g)
capability = float(gpus[0][2])
if capability < 7.5:
    raise RuntimeError(
        f"GPU {gpus[0][1]} (compute capability {capability}) non supportata: "
        "scegli T4 x2, non P100"
    )
GPU_NAME = gpus[0][1]
print(f"Uso cuda:0 = {GPU_NAME}. Le altre GPU restano libere per un secondo processo.")

# ── Clone al commit bloccato ──
if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Commit:", COMMIT)

# ── venv isolato (come il notebook ufficiale): niente pacchetti del kernel ──
PYTHON = REPO / ".venv/bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip", "-q"])
try:
    run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"], check=False)
except Exception as exc:  # noqa: BLE001 -- l'EGL viene verificato davvero più sotto
    print("apt-get non riuscito (continuo):", exc)

TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([PYTHON, "-m", "pip", "install", "-q", "torch==2.9.1", "torchvision==0.24.1",
     "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", str(REPO)])
run([PYTHON, "-m", "pip", "install", "-q", "matplotlib"])
run([PYTHON, "-m", "pip", "check"])

# ── EGL NVIDIA per il rendering headless (replay MP4), copiato dal notebook ufficiale ──
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"


def configure_nvidia_egl(session, candidates):
    library = next((Path(c).resolve() for c in candidates
                    if Path(c).name.startswith("libEGL_nvidia.so.") and Path(c).is_file()), None)
    if library is None:
        raise RuntimeError("libEGL_nvidia non trovata: serve un runtime GPU NVIDIA")
    cfg = Path(session) / "nvidia-egl.json"
    cfg.write_text(json.dumps({"file_format_version": "1.0.0",
                               "ICD": {"library_path": str(library)}}, indent=2) + "\n")
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(cfg.resolve())
    d = str(library.parent)
    rest = [e for e in os.environ.get("LD_LIBRARY_PATH", "").split(":") if e and e != d]
    os.environ["LD_LIBRARY_PATH"] = ":".join([d, *rest])
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    return library


print("EGL:", configure_nvidia_egl(WORK, (
    "/usr/lib64-nvidia/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.0",
    "/usr/lib/nvidia/libEGL_nvidia.so.0",
)))

# ── Il check del repo: CUDA vero, kernel Warp compilato, versioni ──
run([PYTHON, "tools/colab.py", "check", "--repo", REPO, "--out", OUT / "runtime.json"], cwd=REPO)
runtime = json.loads((OUT / "runtime.json").read_text())
print(json.dumps(runtime, indent=2))
run([PYTHON, "scripts/train.py", "--list"], cwd=REPO)
print("\nSETUP OK. Repo:", REPO, "| risultati in:", OUT)


In [ ]:
# @title 2. Smoke test (~5 min): gira? quanto va veloce? quanta memoria?
import json
import os
import re
import signal
from collections import deque
import subprocess
import threading
import time
from datetime import datetime, timezone

ITERATIONS = 12       # poche: le prime 2-3 includono la compilazione dei kernel Warp
TIMEOUT_S = 300       # per run; oltre, SIGINT e si passa al prossimo
WARMUP = 3            # iterazioni scartate nella media

# (etichetta, task, num_envs, argomenti extra). ref_free_jump = fisica del salto a
# 200 Hz, la stessa della task scalinata; flat+rough = costo del terreno a box.
RUNS = [
    ("salto 200Hz, 1024 env", "jumper.ref_free_jump", 1024, []),
    ("salto 200Hz, 2048 env", "jumper.ref_free_jump", 2048, []),
    ("terreno box, 2048 env", "jumper.flat", 2048, ["--scene", "rough"]),
]

ANSI = re.compile(r"\x1b\[[0-9;]*m")
STEPS = re.compile(r"Steps per second:\s*([0-9.]+)")
ITER = re.compile(r"Iteration time:\s*([0-9.]+)s")
LOGDIR = re.compile(r"\[mjrl\] log directory (.*)")


def gpu_memory_mib(index=0):
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits",
         "-i", str(index)], text=True)
    return int(out.strip().splitlines()[0])


def measure(label, task, num_envs, extra):
    cmd = [str(PYTHON), "scripts/train.py", "--task", task, "--backend", "warp",
           "--device", "cuda:0", "--num_envs", str(num_envs),
           "--max-iterations", str(ITERATIONS), "--headless", "--no-tensorboard", *extra]
    print("\n" + "=" * 78 + f"\n{label}\n+ " + " ".join(cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    peak = {"mib": 0}
    stop = threading.Event()

    def poll():
        while not stop.is_set():
            try:
                peak["mib"] = max(peak["mib"], gpu_memory_mib(0))
            except Exception:  # noqa: BLE001
                pass
            stop.wait(2.0)

    threading.Thread(target=poll, daemon=True).start()
    steps, iters, logdir, tail, first_iter_at = [], [], None, deque(maxlen=40), None
    try:
        for line in proc.stdout:
            clean = ANSI.sub("", line)
            tail.append(clean.rstrip())
            m = LOGDIR.search(clean)
            if m:
                logdir = m.group(1).strip()
            m = STEPS.search(clean)
            if m:
                steps.append(float(m.group(1)))
                if first_iter_at is None:
                    first_iter_at = time.time() - t0
                print(f"  it {len(steps):2d}: {steps[-1]:8.0f} env-step/s"
                      f"   GPU {peak['mib']} MiB   t={time.time() - t0:5.0f}s", flush=True)
            m = ITER.search(clean)
            if m:
                iters.append(float(m.group(1)))
            if time.time() - t0 > TIMEOUT_S and proc.poll() is None:
                print("  timeout: interrompo (SIGINT)", flush=True)
                os.killpg(proc.pid, signal.SIGINT)
                break
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
    finally:
        stop.set()
        proc.stdout.close()
    ok = len(steps) > WARMUP
    if not ok:
        print("  *** nessuna iterazione completata. Ultime righe:\n   " + "\n   ".join(tail))
    s = steps[WARMUP:] if ok else steps
    it = iters[WARMUP:] if len(iters) > WARMUP else iters
    return {
        "label": label, "task": task, "num_envs": num_envs, "extra": extra,
        "ok": ok, "returncode": proc.returncode,
        "env_steps_per_s": round(sum(s) / len(s)) if s else None,
        "iteration_s": round(sum(it) / len(it), 2) if it else None,
        "build_s": round(first_iter_at - (iters[0] if iters else 0), 1) if first_iter_at else None,
        "gpu_peak_mib": peak["mib"], "wall_s": round(time.time() - t0, 1),
        "log_dir": logdir, "tail": list(tail)[-15:],
    }


results = [measure(*r) for r in RUNS]

# ── Tabella e proiezione ──
print("\n" + "=" * 78)
print(f"GPU: {GPU_NAME} | commit {COMMIT[:9]} | {datetime.now(timezone.utc):%Y-%m-%d %H:%M} UTC")
print(f"{'run':<26}{'env-step/s':>12}{'s/iter':>9}{'build s':>9}{'GPU MiB':>9}{'iter/h':>9}{'h/6000 it':>11}")
for r in results:
    if r["ok"] and r["iteration_s"]:
        per_h = 3600 / r["iteration_s"]
        print(f"{r['label']:<26}{r['env_steps_per_s']:>12}{r['iteration_s']:>9}"
              f"{r['build_s'] or 0:>9}{r['gpu_peak_mib']:>9}{per_h:>9.0f}{6000 / per_h:>11.1f}")
    else:
        print(f"{r['label']:<26}{'FALLITO':>12}  (returncode {r['returncode']})")
print("\nUna sessione Kaggle da 10.5 h di training copre, per il run 'salto 2048 env':")
ref = next((r for r in results if r["num_envs"] == 2048 and r["task"].endswith("jump") and r["ok"]), None)
if ref and ref["iteration_s"]:
    print(f"  ~{10.5 * 3600 / ref['iteration_s']:.0f} iterazioni  (ref_free_jump ne ha richieste 6000;"
          f" la scalinata stimata 10-20k)")

(OUT / "smoke.json").write_text(json.dumps(
    {"gpu": GPU_NAME, "gpus": gpus, "commit": COMMIT, "runtime": runtime, "results": results},
    indent=2))
print("\nSalvato:", OUT / "smoke.json", "→ mandami questo file (o incolla la tabella).")


## Come leggere i numeri

- **env-step/s**: passi di simulazione al secondo, media delle iterazioni dopo la compilazione. È il numero che decide quante sessioni servono.
- **s/iter**: una iterazione PPO = `num_envs × 48` passi (le task di salto a 200 Hz). `iter/h` e `h/6000 it` sono la proiezione.
- **GPU MiB**: picco di memoria. Sotto ~12 000 MiB su T4 c'è margine per raddoppiare `num_envs`; sopra 14 000 no.
- **build s**: tempo prima della prima iterazione (compilazione kernel Warp + costruzione scena). Si paga una volta per run.
- Il run **"terreno box"** dice quanto costa in più il terreno a gradini rispetto al piano: la task scalinata sarà circa a metà strada tra il secondo e il terzo run.

Niente di questo dice se una policy *impara*: è solo la velocità del tapis roulant. Il notebook di training vero e proprio arriva con la task.
